In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file


In [2]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain_fireworks import ChatFireworks
from langchain_groq import ChatGroq


In [3]:
model = ChatGroq(model_name="llama3-groq-70b-8192-tool-use-preview", temperature=0)
# model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, if not explicitly provided do not guess. Extract partial info"),
    ("human", "{input}")
])

In [5]:
from typing import List, Optional
from pydantic import BaseModel, Field

class PersonalInfo(BaseModel):
    """Model representing personal information."""
    name: str = Field(..., description="The name of the individual.")
    role: Optional[str] = Field(None, description="The current role or position of the individual.")
    cofo: Optional[str] = Field(None, description="Contact information, such as email or phone number.")
    location: Optintact_inonal[str] = Field(None, description="Current location of the individual.")

class Project(BaseModel):
    """Model representing a project the individual is working on."""
    title: str = Field(..., description="The title of the project.")
    description: Optional[str] = Field(None, description="A brief description of the project.")
    technologies: List[str] = Field(..., description="Technologies being used in the project.")
    milestones: List[str] = Field(..., description="Key milestones or events related to the project.")
    collaborators: Optional[List[str]] = Field(None, description="List of collaborators on the project.")

class Publication(BaseModel):
    """Model representing a publication related to the individual's work."""
    title: str = Field(..., description="The title of the publication.")
    date: str = Field(..., description="The date of publication.")
    focus_area: List[str] = Field(..., description="Focus areas covered in the publication.")
    url: Optional[str] = Field(None, description="URL link to the publication for reference.")

class KeyEvent(BaseModel):
    """Model representing a key event in the individual's career."""
    event: str = Field(..., description="A brief description of the key event.")
    date: str = Field(..., description="The date when the event occurred.")
    details: Optional[str] = Field(None, description="Additional details about the event.")

class SentimentAnalysis(BaseModel):
    """Model for tracking sentiment regarding projects or events."""
    project_title: str = Field(..., description="The title of the project or event being analyzed.")
    sentiment: str = Field(..., description="The overall sentiment (e.g., positive, negative, neutral).")
    feedback: Optional[str] = Field(None, description="Additional feedback or comments regarding the sentiment.")
    score: Optional[float] = Field(None, description="Numerical score representing the sentiment strength.")

class PreferenceTrend(BaseModel):
    """Model for tracking preferences over time."""
    technology: str = Field(..., description="The technology or tool being tracked.")
    platform: str = Field(..., description="The platform being tracked.")
    network: str = Field(..., description="The network being tracked.")

class ChatHistory(BaseModel):
    """Model representing the overall chat history including personal info, projects, publications, events, and trends."""
    personal_info: PersonalInfo = Field(..., description="Personal information of the individual.")
    projects: List[Project] = Field(..., description="A list of projects the individual is currently working on.")
    publications: List[Publication] = Field(..., description="A list of publications by the individual.")
    key_events: List[KeyEvent] = Field(..., description="Key events in the individual's career.")
    sentiment_analysis: List[SentimentAnalysis] = Field(..., description="Sentiment analysis results for various projects.")
    preference_trends: List[PreferenceTrend] = Field(..., description="Trends in preferences regarding technologies and projects.")
    notes: Optional[str] = Field(None, description="Any additional notes or comments related to the chat history.")


In [6]:
# Read chat_history.txt file into chat_history with ut8
chat_history = open("chat_history.txt", "r", encoding="utf-8").read()
print(chat_history)


2024-01-15 09:00 AM - Phi: Hey, I just got an offer for an AI Engineer Intern position at NeurondAI!
2024-01-15 09:02 AM - Alex: Congrats Phi! When do you start?
2024-01-15 09:05 AM - Phi: I’ll be starting on September 9th, 2024. Really excited!
2024-01-15 09:10 AM - Alex: That’s awesome! What kind of work will you be doing there?
2024-01-15 09:15 AM - Phi: I’ll be working on AI backend systems, building services using Docker and FastAPI.
2024-01-20 11:00 AM - Phi: I’m currently working on a side project called Phinx, where I’m building an AI backend image.
2024-01-20 11:15 AM - Alex: Nice! What tools are you using for that?
2024-01-20 11:30 AM - Phi: I’m using Docker to build the image and integrating FastAPI for some of the services. I’m also using Playwright for testing.
2024-02-10 08:00 AM - Phi: By the way, I’ve been focusing on OCR lately. I’m digitizing Vietnamese text from scanned documents to help automate data extraction.
2024-02-10 08:15 AM - Alex: That sounds really useful

In [7]:
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser
from pprint import pprint
Phinx_extraction_function = [
    convert_to_openai_function(ChatHistory)
]

extraction_model = model.bind(
    functions=Phinx_extraction_function,
    function_call={"name": "ChatHistory"}
)


# Define the extraction chain to include the JsonOutputFunctionsParser
extraction_chain_full = prompt | extraction_model | JsonOutputFunctionsParser()

# Invoke the extraction chain with the chat history input
full_result = extraction_chain_full.invoke({"input": chat_history})
# pprint(full_result)


In [9]:
import json

# Dump full_result to a JSON file
with open('full_result.json', 'w', encoding='utf-8') as f:
    json.dump(full_result, f, ensure_ascii=False, indent=4)

print("Full result has been saved to 'full_result.json'")


Full result has been saved to 'full_result.json'
